# Evidence of a case: upload, convert, split

**What it does**
Puts evidence files into the evidence folder of one case and prepares them exactly as the assessment will send them to the AI:
1. **Upload**: files go to `storage/evidences/<case>/` (any subfolders), the same as the API's `POST /api/v1/cases/{case}/evidences`.
2. **Units**: one unit per uploaded file. An email is one unit with all its attachments, so the AI sees the approval in the email next to the log attached to it.
3. **Convert**: Word becomes PDF, Excel becomes text and pictures, email becomes text with its attachments converted (notebooks 07, 08 and 09 show each converter).
4. **Split**: a unit too big for one AI call is cut into parts:
   - **PDF**: by pages (`KCT_PDF_MAX_PAGES`, 2 pages of overlap) or size (`KCT_PDF_MAX_MB`)
   - **Excel**: by sheet, then row blocks, with the header repeated in every block
   - **Email**: the email text goes with every part

**Why**
The assessment loops over these parts: one Gemini call per part, with the test steps and the control details and test plan. This notebook shows the loop's input before anything is sent: how many calls a case needs and roughly how many tokens each takes.

**How to use**
1. Run Setup. It picks the newest case; set `CASE` to use another one. No case yet? Run `04_demo_data.ipynb` first.
2. Step 1 puts fake evidence of every kind into the case, including a 120-page PDF so you can see the splitting. Skip it when the case already has real evidence.
3. Run the rest. Change the limits in `.env` (or Step 3 here) and run Step 4 again to see the effect.

**Needs**: the evidence packages (`python-docx`, `reportlab`, `openpyxl`, `extract-msg`, `pypdf`; Step 3 of `01_setup.ipynb`).

## Setup

Finds the project folder, reads `.env`, and picks the case.

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from services.app import boot
from services.config import settings

kct = boot()
cases = kct.files.case_ids()
CASE = cases[-1] if cases else None     # or e.g. "case-002"

if CASE is None:
    print("❌ No case yet: run 04_demo_data.ipynb first")
else:
    evidence = kct.evidences.case(CASE)
    print("Case:            ", CASE, f"(cases: {', '.join(cases)})")
    print("Evidence folder: ", evidence.root)

## Step 1: Put fake evidence into the case (optional)

Stores one file of every kind with the factories, as an upload would:
- Word (`DocxFactory`)
- Excel with 300 rows (`ExcelFactory`), in a subfolder `excel/`
- an email thread with attachments (`MsgFactory`)
- a 120-page PDF (`PdfFactory`)

Same names overwrite the earlier files.

In [ ]:
from factories import DocxFactory, ExcelFactory, MsgFactory, PdfFactory
from services.files.file_service import UploadedFile

uploads = [
    ("", DocxFactory.name, DocxFactory().make()),
    ("excel", ExcelFactory.name, ExcelFactory(rows=300).make()),
    ("", MsgFactory.name, MsgFactory(history=4, images=3).make()),
    ("", PdfFactory.name, PdfFactory(pages=120).make()),
]
for folder, name, data in uploads:
    for stored in evidence.store([UploadedFile(name=name, data=data)], folder=folder):
        print(f"✅ {stored.path}  ({stored.size / 1024:.0f} KB{', replaced' if stored.replaced else ''})")

## Step 2: Evidence files of the case

What was uploaded, without the files the converters make next to them.

In [ ]:
for relative in evidence.evidence_files():
    print(f"{str(relative):<60} {evidence.path(relative).stat().st_size / 1024:>8.1f} KB")

## Step 3: Split limits

The limits from `.env` (or the defaults). Change them here to try other values; this changes only this notebook.

In [ ]:
from services.evidences.evidence_splitter import EvidenceSplitter

config = settings()
MAX_PAGES = config.pdf_max_pages        # KCT_PDF_MAX_PAGES
MAX_MB = config.pdf_max_mb              # KCT_PDF_MAX_MB
OVERLAP = config.pdf_overlap_pages      # KCT_PDF_OVERLAP_PAGES
MAX_TOKENS = config.evidence_max_tokens # KCT_EVIDENCE_MAX_TOKENS

evidence.splitter = EvidenceSplitter(MAX_PAGES, MAX_MB, OVERLAP, MAX_TOKENS)
print(f"PDF: {MAX_PAGES} pages or {MAX_MB} MB per part, {OVERLAP} pages overlap; {MAX_TOKENS:,} tokens per part")

## Step 4: Units and parts (the assessment's loop)

Converts every file and splits it. Each line is one future AI call. The documents listed under a part are what the AI gets in that call, besides the test steps, control details and test plan. Converting takes a few seconds per file.

In [ ]:
units = evidence.units()
parts = evidence.parts(units)

for unit in units:
    unit_parts = [p for p in parts if p.unit == str(unit.path)]
    print(f"\n{unit.path}  [{unit.kind}]  -> {len(unit_parts)} part(s)")
    for note in unit.notes:
        print("   note:", note)
    for part in unit_parts:
        print(f"   part {part.number}/{part.total}: ~{part.estimated_tokens:,} tokens")
        for doc in part.documents:
            print(f"      {doc.title[:110]}")

print(f"\n{len(units)} evidence file(s) -> {len(parts)} AI call(s), ~{sum(p.estimated_tokens for p in parts):,} evidence tokens in total")

## Step 5: As a table (needs pandas)

In [ ]:
import pandas as pd

pd.DataFrame([
    {"unit": p.unit, "part": f"{p.number}/{p.total}", "documents": len(p.documents), "estimated tokens": p.estimated_tokens}
    for p in parts
])